# Phishing Email Detection with BERT
## Notebook 05 – Gmail Integration and Live Email Scoring

This notebook connects the trained BERT phishing email classifier to the Gmail API.
It fetches recent inbox messages, cleans and normalizes their content, runs the model
to obtain phishing probabilities, and summarizes results for manual review.


### Notebook objectives

- Authenticate with the Gmail API using OAuth2 credentials.
- Fetch a recent subset of inbox messages that are likely to be important (non-social, non-promotion).
- Extract and normalize email bodies (plain text or HTML) into a BERT-ready text representation.
- Load the fine-tuned phishing detection model and tokenizer from disk.
- Score each email, compute phishing probabilities, and assign labels (Ham / Phishing / Uncertain).
- Heuristically flag likely marketing/notification emails using unsubscribe signals.
- Display a compact summary DataFrame and optionally export detailed predictions to CSV.


### 1. Imports and configuration

Import Gmail API utilities, text-cleaning helpers, and model loading tools.


In [1]:
import os, re, html, base64
from pathlib import Path
from typing import List, Optional
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from bs4 import BeautifulSoup
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.auth.transport.requests import Request

SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']
CREDENTIALS_PATH = 'credentials.json'
TOKEN_PATH = 'token.json'

QUERY = 'in:inbox newer_than:14d -category:promotions -category:social'

MAX_RESULTS = 50

MODEL_DIR = '../models/saved_model'
LABELS = {0: "Ham", 1: "Phishing"}
THRESH_PHISH = 0.85
THRESH_HAM = 0.15
MAX_BODY_CHARS = 3000
EXPORT_CSV = True
EXPORT_PATH = '../analysis/gmail_prediction/gmail_predictions.csv'


### 2. Text normalization and HTML cleanup utilities

Define helper functions that clean and normalize raw email text, ensuring consistent
input formatting


In [2]:
# Normalize spacing and repair broken URL punctuation patterns
def normalize_punct_spacing_for_urls(s):
    s = str(s)
    s = re.sub(r'(?i)([a-z0-9])((?:https?|ftp|file))', r'\1 \2', s)
    s = re.sub(r'\s*\.\s*', '.', s)
    s = re.sub(r'\s*/\s*', '/', s)
    s = re.sub(r'\s*:\s*', ':', s)
    s = re.sub(r'\b(?:h\s*t\s*t\s*p(?:s)?)\b', lambda m: m.group(0).replace(' ', ''), s, flags=re.IGNORECASE)
    s = re.sub(r'\b(?:f\s*t\s*p)\b', lambda m: m.group(0).replace(' ', ''), s, flags=re.IGNORECASE)
    s = re.sub(r'(?i)\b(https?|ftp|file)\s*[\.:;]\s*/\s*/', r'\1://', s)
    return s


# Strip HTML tags and extract visible text content
def strip_html(html_text: str) -> str:
    soup = BeautifulSoup(html_text, 'html.parser')
    for tag in soup(['script', 'style']):
        tag.decompose()
    text = soup.get_text(separator=' ', strip=True)
    return ' '.join(text.split())


# Clean, normalize, and prepare raw email text for BERT input
def clean_text_for_bert(text: str) -> str:
    if text is None:
        return ""
    text = html.unescape(str(text))
    text = normalize_punct_spacing_for_urls(text)
    return re.sub(r"\s+", " ", text).strip()


### 3. Gmail authentication
Authenticate using OAuth, refresh tokens if needed, and build the Gmail service object.


In [3]:
# Authenticate with Gmail using OAuth2 and return a Gmail API service object
def gmail_service_popup():
    creds = None

    if os.path.exists(TOKEN_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)

    if not creds or not creds.valid:
        if creds and getattr(creds, "refresh_token", None):
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_PATH, SCOPES)
            try:
                creds = flow.run_local_server(port=0, open_browser=True)
            except Exception:
                print("Could not launch browser. Please open the displayed URL manually.")
                creds = flow.run_console()

        with open(TOKEN_PATH, 'w') as f:
            f.write(creds.to_json())

    return build('gmail', 'v1', credentials=creds)


service = gmail_service_popup()


### 4. Gmail message retrieval and payload extraction

Define helper functions to decode Base64 payloads, extract header fields, and
recursively pull plain text or cleaned HTML content from Gmail message bodies.


In [4]:
# Decode a Base64URL-encoded Gmail message body
def base64url_decode(data: str) -> bytes:
    return base64.urlsafe_b64decode(data + '==')

# Retrieve a specific email header value by name
def get_header(headers: List[dict], name: str) -> Optional[str]:
    for h in headers:
        if h.get('name') == name:
            return h.get('value')
    return None

# Extract plain text or HTML-cleaned content from a Gmail message payload
def extract_plain_text_from_payload(payload: dict) -> str:
    texts = []

    # Recursively process MIME parts
    def handle_part(p):
        mime = p.get('mimeType', '')
        body = p.get('body', {})
        data = body.get('data')

        # Handle nested multipart MIME structure
        if 'parts' in p:
            for sub in p['parts']:
                handle_part(sub)
        else:
            if data:
                content = base64url_decode(data).decode('utf-8', errors='ignore')
                if mime.startswith('text/plain'):
                    texts.append(content)
                elif mime.startswith('text/html'):
                    texts.append(strip_html(content))

    # Process top-level multipart emails
    if 'parts' in payload:
        for part in payload['parts']:
            handle_part(part)
    else:
        data_root = payload.get('body', {}).get('data')
        mime_root = payload.get('mimeType', '')
        if data_root:
            content = base64url_decode(data_root).decode('utf-8', errors='ignore')
            if mime_root.startswith('text/plain'):
                texts.append(content)
            elif mime_root.startswith('text/html'):
                texts.append(strip_html(content))

    return "\n\n".join([t for t in texts if t]).strip()


### 5 Fetch message IDs from Gmail

This cell sends the Gmail API request using the search query and retrieves the
list of message IDs.


In [5]:
try:
    resp = service.users().messages().list(
        userId='me', q=QUERY, maxResults=MAX_RESULTS
    ).execute()
except HttpError as e:
    print("Gmail API error:", e)
    raise

messages = resp.get('messages', []) or []
print(f"Found {len(messages)} messages for query: {QUERY}")


Found 50 messages for query: in:inbox newer_than:14d -category:promotions -category:social


### 6. Extract message subject, body, and metadata

Loop through each Gmail message ID, fetch the full message, extract headers and body text, clean the content, and store structured fields (subject, sender, unsubscribe info, and processed body) into a DataFrame.


In [6]:
rows = []
for m in messages:
    msg = service.users().messages().get(userId='me', id=m['id'], format='full').execute()
    payload = msg.get('payload', {})
    headers = payload.get('headers', [])
    subj = get_header(headers, 'Subject') or ''
    frm  = get_header(headers, 'From') or ''
    list_unsub = get_header(headers, 'List-Unsubscribe') or ''
    has_list_unsub = bool(list_unsub.strip())

    body = extract_plain_text_from_payload(payload)
    body = body[:MAX_BODY_CHARS] if MAX_BODY_CHARS else body
    body_clean = clean_text_for_bert(body)
    has_unsubscribe_word = 'unsubscribe' in body_clean.lower()

    rows.append({
        "id": m["id"],
        "from": frm,
        "subject": clean_text_for_bert(subj),
        "body": body_clean,
        "list_unsubscribe": list_unsub,
        "has_list_unsubscribe": has_list_unsub,
        "has_unsubscribe_word": has_unsubscribe_word,
        "is_promotion": True,
    })

df = pd.DataFrame(rows)
print(f"Fetched {len(df)} messages.")


Fetched 50 messages.


### 7. Load model and tokenizer

Load the fine-tuned BERT model and tokenizer from the saved directory


In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'   # Select GPU if available

try:
    # Try loading model/tokenizer strictly from local directory
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR,
        local_files_only=True
    ).to(device).eval()                                    

except Exception as e:
    # Fallback
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR
    ).to(device).eval()



2025-12-09 12:36:53.386549: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-09 12:36:53.916056: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-09 12:36:56.474236: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### 8. Score emails with the BERT model

Compute phishing probabilities and model logits for each email

In [8]:
@torch.inference_mode()
# Score a list of email texts and return phishing probabilities and logits
def score_batch(texts):
    probs_out, logits_out = [], []

    for t in tqdm(texts, desc="Scoring emails"):
        enc = tokenizer(
            t,
            truncation=True,
            max_length=512,
            padding=False,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits[0]
        probs = torch.softmax(logits, dim=-1).cpu().tolist()
        probs_out.append(probs[1])
        logits_out.append(float(logits[1].item()))

    return probs_out, logits_out


p_list, logit_list = score_batch(df["body"].tolist())
df["p_phish"] = p_list
df["logit_phish"] = logit_list


Scoring emails: 100%|███████████████████████████████████████████████████████████████████| 50/50 [00:01<00:00, 29.71it/s]


### 9. Classification and summary statistics

Convert phishing probabilities into discrete labels (Ham / Phishing / Uncertain) and print the distribution of predictions.

In [9]:
# Convert a probability into a discrete label (Phishing / Ham / Uncertain)
def classify(p):
    if p >= THRESH_PHISH: 
        return "Phishing"
    if p <= THRESH_HAM: 
        return "Ham"
    return "Uncertain"

df["pred"] = df["p_phish"].apply(classify)


### 10. Display filtered results

Show a summary table of email predictions

In [10]:
df["likely_marketing_ham"] = (
    df["has_list_unsubscribe"] &
    (df["p_phish"] < 0.5)
)

view_cols = [
    "from", "subject", "p_phish", "logit_phish",
    "pred", "has_list_unsubscribe",
    "has_unsubscribe_word", "likely_marketing_ham",
]

df_view = df[view_cols].reset_index(drop=True)
df_view


,from,subject,p_phish,logit_phish,pred,has_list_unsubscribe,has_unsubscribe_word,likely_marketing_ham
0,LinkedIn Job Alerts <jobalerts-noreply@linkedi...,“Software Engineer”:Uber - Software Engineer I...,0.371303,-0.827042,Uncertain,True,False,True
1,Indeed <alert@indeed.com>,Mackenzie Financial Corporation is hiring for ...,0.006553,-3.034491,Ham,True,False,True
2,LinkedIn Job Alerts <jobalerts-noreply@linkedi...,“Cyber Security Analyst”:TD - Information Secu...,0.965544,2.030736,Phishing,True,False,False
3,LinkedIn <updates-noreply@linkedin.com>,George Collins recently posted,0.001291,-3.966023,Ham,True,False,True
4,Slidesgo <info@slidesgo.com>,Welcome to Slidesgo,0.863505,1.047016,Phishing,False,False,False
5,Freepik <noreply@freepik.com>,Account activation,0.005880,-3.110328,Ham,False,False,False
6,Simran Panchal <simran.panchal@usepepper.com>,Invitation from an unknown sender:Product Cont...,0.031826,-2.205475,Ham,False,False,False
7,Indeed <invitetoapply@match.indeed.com>,IT Help Desk Co-op @ MatcorMatsu,0.012179,-2.726140,Ham,True,True,True
8,Atlassian Careers <atlassian@nurture.icims.com>,Welcome to the Atlassian Talent Community!,0.995749,3.288253,Phishing,False,False,False
9,no-reply@us.greenhouse-mail.io,Thank you for your application for Data Engine...,0.001381,-3.916281,Ham,False,False,False


### 11. Print the results

Assumption: Because this dataset is pulled from a personal inbox, all retrieved emails are expected to be legitimate (ham). Therefore, any model prediction of phishing is treated as a false positive.
If the query instead retrieves messages from the spam folder, then ground-truth labels must be manually assigned before evaluation.

In [11]:
# Count total Ham and Phishing predictions
num_ham = (df["pred"] == "Ham").sum()
num_phishing = (df["pred"] == "Phishing").sum()
num_uncertain = (df["pred"] == "Uncertain").sum()

false_positives = num_phishing       
false_negatives = 0                 

print("=== Prediction Summary ===")
print(f"Total Emails Scanned: {len(df)}")
print(f"Ham Predictions:       {num_ham}")
print(f"Phishing Predictions:  {num_phishing}")
print(f"Uncertain Predictions:  {num_uncertain}")
print(f"False Positives (FP):  {false_positives}")
print(f"False Negatives (FN):  {false_negatives}")


=== Prediction Summary ===
Total Emails Scanned: 50
Ham Predictions:       28
Phishing Predictions:  18
Uncertain Predictions:  4
False Positives (FP):  18
False Negatives (FN):  0


### Redact personal infomation

Create privacy-safe text fields by removing emails, URLs, phone numbers, and limiting preview length.

In [12]:
import re

def redact(text: str) -> str:
    if not isinstance(text, str):
        return text
    
    # Redact email addresses
    text = re.sub(
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",
        "[EMAIL]",
        text,
    )
    
    # Redact URLs
    text = re.sub(
        r"http[s]?://\S+",
        "[URL]",
        text,
    )
    
    # Redact phone numbers
    text = re.sub(
        r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b",
        "[PHONE]",
        text,
    )
    
    return text
    
df["redacted_body"] = df["body"].apply(redact)

# Shorten subject line safely
df["subject_short"] = df["subject"].astype(str).str.slice(0, 50) + "..."
df["subject_short"] = df["subject_short"].fillna("(no subject)")

# Create a short preview of the redacted body
df["preview"] = df["redacted_body"].astype(str).str.slice(0, 50) + "..."
df["preview"] = df["preview"].fillna("(no content)")

print("Redaction complete. Added fields: redacted_body, subject_short, preview.")


Redaction complete. Added fields: redacted_body, subject_short, preview.


### 12. Export predictions to CSV


In [13]:
if EXPORT_CSV:
    os.makedirs(os.path.dirname(EXPORT_PATH), exist_ok=True)
    df[["id", "from", "subject_short", "preview", "pred", "p_phish", "logit_phish"]].to_csv(
        EXPORT_PATH, index=False, encoding="utf-8"
    )
    print(f"Saved privacy-safe predictions to {EXPORT_PATH}")


Saved privacy-safe predictions to ../analysis/gmail_prediction/gmail_predictions.csv
